# TRACE M3: model pipeline smoke on a free GPU

Runtime > Change runtime type > **T4 GPU**. Run the cells top to bottom. Upload `trace_vr_src.zip` when the second cell asks for it.

This notebook checks that the pipeline works and measures cost. It does not produce research results.

In [ ]:
import subprocess, torch, sys, platform
print(sys.version)
print(platform.platform())
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
import os, zipfile
if not os.path.exists('/content/trace/src/trace_vr'):
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    os.makedirs('/content/trace', exist_ok=True)
    zipfile.ZipFile(name).extractall('/content/trace')
print(sorted(os.listdir('/content/trace')))

In [ ]:
%cd /content/trace
!pip -q install 'transformers==5.22.2' 'peft==0.21.2' 'accelerate==1.15.0' pyyaml pytest networkx 2>&1 | tail -2
import transformers, peft, PIL
print('transformers', transformers.__version__, 'peft', peft.__version__, 'pillow', PIL.__version__)

In [ ]:
%cd /content/trace
%env PYTHONPATH=src
!python -m trace_vr doctor
!python -m pytest -q 2>&1 | tail -2

In [ ]:
# Rebuild the pilot data here and compare with the manifest built on the laptop.
# content digest must match; images digest may differ with the Pillow build.
%cd /content/trace
import json, shutil
shutil.copy('manifests/pilot.json', '/content/pilot_laptop.json')
!python -m trace_vr generate --config configs/data_pilot.yaml | tail -4
a, b = json.load(open('/content/pilot_laptop.json')), json.load(open('manifests/pilot.json'))
print('content digest identical:', a['content_digest'] == b['content_digest'])
print('images digest identical: ', a['images_digest'] == b['images_digest'])
!python -m trace_vr validate-data --manifest manifests/pilot.json | head -1

In [ ]:
# Frozen base model on 10 dev root graphs: pipeline checks, speed, memory.
%cd /content/trace
!python scripts/model_smoke.py manifests/pilot.json dev 10 runs/a0_smoke_t4 2>&1 | grep -v 'Loading weights' | tail -45

In [ ]:
# Tiny LoRA overfit on 16 repair samples (wiring check, not a result).
%cd /content/trace
!python scripts/overfit_smoke.py manifests/pilot.json train 16 96 runs/overfit_t4 2>&1 | grep -v 'Loading weights' | tail -30

In [ ]:
%cd /content/trace
!cat runs/a0_smoke_t4/metrics.json | head -40
!cat runs/overfit_t4/metrics.json
!cd /content/trace && zip -qr /content/m3_runs.zip runs -x 'runs/*/adapter/*' && ls -la /content/m3_runs.zip